# ELSST Track 1: RAG Pipeline

This notebook implements two RAG variants for Track
**Standard RAG** retrieves the most similar labeled training examples per document and uses them as dynamic in-context examples (unlike Section 5's two fixed Examples, retrieved per-document via embedding
similarity) run on a 100 document sample as a lightweight ablation **Ontology-Grounded RAG (OG-RAG)** retrieves each candidate's ELSST hierarchy context broader/narrower
concepts and provides it as structured grounding for the LLM's reasoning run on the
full 756 validation documents as the primary result, since it's the more novel and
central contribution of this section.

## 1. Environment Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, json, time
import numpy as np

project_path = '/content/drive/MyDrive/ELSST_Project'
RESULTS_DIR = os.path.join(project_path, 'results')
CHECKPOINT_DIR = os.path.join(project_path, 'checkpoints')

!pip install google-genai datasets huggingface_hub -q

from google.colab import userdata
from google import genai

api_key = userdata.get('GEMINI_API_KEY')
client = genai.Client(api_key=api_key)
MODEL_NAME = "gemini-3.5-flash-lite"

from datasets import load_from_disk
from huggingface_hub import hf_hub_download

dataset = load_from_disk(os.path.join(project_path, 'dataset'))

pool_path = hf_hub_download(repo_id="JohnWang10086/elsst-track1", filename="concept_pool.jsonl", repo_type="dataset")
concept_pool_lookup = {}
with open(pool_path, "r") as f:
    for line in f:
        c = json.loads(line)
        concept_pool_lookup[c["concept_id"]] = c

val_ids = [ex['id'] for ex in dataset['validation']]
val_texts = [ex['text'] for ex in dataset['validation']]
gold = {ex['id']: ex['retrieval_labels']['positive_ids'] for ex in dataset['validation']}

with open(os.path.join(RESULTS_DIR, 'qwen_predictions.json')) as f:
    qwen_predictions = json.load(f)

with open(os.path.join(RESULTS_DIR, 'concept_hierarchy_strict.json')) as f:
    concept_neighbors_raw = json.load(f)
concept_neighbors = {cid: set(neighbors) for cid, neighbors in concept_neighbors_raw.items()}

print(f"Loaded {len(concept_pool_lookup)} concepts, {len(val_ids)} val docs, hierarchy for {len(concept_neighbors)} concepts")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


concept_pool.jsonl:   0%|          | 0.00/800k [00:00<?, ?B/s]

Loaded 3433 concepts, 756 val docs, hierarchy for 3433 concepts


In [ ]:
def reciprocal_rank(ranked_ids, positive_ids):
    positive_set = set(positive_ids)
    for rank, cid in enumerate(ranked_ids, start=1):
        if cid in positive_set:
            return 1.0 / rank
    return 0.0

def recall_at_k(ranked_ids, positive_ids, k):
    if not positive_ids:
        return 0.0
    top_k = set(ranked_ids[:k])
    return len(top_k & set(positive_ids)) / len(positive_ids)

def ndcg_at_k(ranked_ids, positive_ids, k=10):
    positive_set = set(positive_ids)
    dcg = sum(1.0 / np.log2(i + 1) for i, cid in enumerate(ranked_ids[:k], start=1) if cid in positive_set)
    ideal_hits = min(len(positive_ids), k)
    idcg = sum(1.0 / np.log2(i + 1) for i in range(1, ideal_hits + 1))
    return dcg / idcg if idcg > 0 else 0.0

def evaluate_retrieval(predictions, gold):
    mrr, r5, r10, ndcg = [], [], [], []
    for doc_id, positive_ids in gold.items():
        ranked = predictions[doc_id]
        mrr.append(reciprocal_rank(ranked, positive_ids))
        r5.append(recall_at_k(ranked, positive_ids, 5))
        r10.append(recall_at_k(ranked, positive_ids, 10))
        ndcg.append(ndcg_at_k(ranked, positive_ids, 10))
    return {"MRR": float(np.mean(mrr)), "Recall@5": float(np.mean(r5)), "Recall@10": float(np.mean(r10)), "NDCG@10": float(np.mean(ndcg))}

def run_or_load(name, compute_fn):
    path = os.path.join(RESULTS_DIR, f"{name}.json")
    if os.path.exists(path):
        print(f"Loaded cached result: {name}")
        with open(path) as f:
            return json.load(f)
    print(f"Running: {name}")
    result = compute_fn()
    with open(path, "w") as f:
        json.dump(result, f, indent=2)
    return result

def call_llm(prompt, max_retries=3, wait_seconds=4):
    for attempt in range(max_retries):
        try:
            response = client.models.generate_content(model=MODEL_NAME, contents=prompt)
            time.sleep(wait_seconds)
            return response.text
        except Exception as e:
            if "429" in str(e):
                backoff = 15 * (attempt + 1)
                print(f"Rate limited, waiting {backoff}s (attempt {attempt + 1})...")
                time.sleep(backoff)
            else:
                raise
    raise RuntimeError("Failed after max retries")

## 2. Standard RAG (Retrieval-Augmented Generation)
> **Approach:** Unlike Section 5's few-shot prompting, which used the same 2 fixed worked
examples for every document, standard RAG retrieves the training examples most similar
to each specific document via embedding similarity and uses those as dynamic in-context
examples. The idea: an example that's actually similar to the current document should
teach the model more than a generic fixed example.
>
> **Scope:** run as a lightweight ablation on a 100 document sample rather than the full 756 this is here as a comparison point against dynamic few-shot's promise, not the primary contribution of this section that's OG-RAG below.
>
> **Hypothesis:** Should modestly outperform Section 5's fixed few-shot, since retrieved examples are contextually relevant rather than arbitrary.

In [ ]:
!pip install sentence-transformers -q

from sentence_transformers import SentenceTransformer

device = "cuda" if __import__('torch').cuda.is_available() else "cpu"
sbert_model = SentenceTransformer("all-MiniLM-L6-v2", device=device)

def chunk_text(text, chunk_size=200, overlap=50):
    """Overlapping word-chunks so we don't silently truncate long documents -
    same fix as the Section 6 SBERT baseline, needed here too since we're
    encoding full training passages for retrieval."""
    words = text.split()
    chunks = []
    start = 0
    while start < len(words):
        chunks.append(' '.join(words[start:start + chunk_size]))
        start += chunk_size - overlap
    return chunks

def embed_document(text, model):
    """Chunk, embed each chunk, mean-pool to get one vector per document -
    simpler than max-pooling since we're finding a similar *document*, not
    matching to a specific concept."""
    chunks = chunk_text(text)
    chunk_embeddings = model.encode(chunks, show_progress_bar=False)
    return np.mean(chunk_embeddings, axis=0)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

### 2.1 Sample and embed
> Sampling 100 validation documents fixed seed for reproducibility and embedding both the full training set and this sample, so we can retrieve similar train examples per document. Cached to Drive since embedding 3,000 documents takes a few minutes.

In [ ]:
def embed_documents_batched(texts, model, chunk_size=200, overlap=50):
    """Encode all chunks across all documents in one large batched call -
    far faster than calling .encode() separately per document, since the
    model is optimized for big batches, not thousands of tiny individual calls."""
    all_chunks = []
    doc_chunk_map = []
    for doc_idx, text in enumerate(texts):
        chunks = chunk_text(text, chunk_size, overlap)
        all_chunks.extend(chunks)
        doc_chunk_map.extend([doc_idx] * len(chunks))

    print(f"Encoding {len(all_chunks)} chunks from {len(texts)} documents in one batch...")
    chunk_embeddings = model.encode(all_chunks, show_progress_bar=True, batch_size=64)

    doc_chunk_map = np.array(doc_chunk_map)
    doc_embeddings = np.zeros((len(texts), chunk_embeddings.shape[1]))
    for doc_idx in range(len(texts)):
        mask = doc_chunk_map == doc_idx
        doc_embeddings[doc_idx] = chunk_embeddings[mask].mean(axis=0)

    return doc_embeddings


def get_or_embed(name, texts):
    path = os.path.join(CHECKPOINT_DIR, f"{name}.npy")
    if os.path.exists(path):
        print(f"Loaded cached embeddings: {name}")
        return np.load(path)
    embeddings = embed_documents_batched(texts, sbert_model)
    np.save(path, embeddings)
    return embeddings

train_embeddings = get_or_embed("rag_train_embeddings", train_texts)
sample_val_embeddings = get_or_embed("rag_sample_val_embeddings", sample_val_texts)

print(f"Train embeddings: {train_embeddings.shape}, Sample val embeddings: {sample_val_embeddings.shape}")

Encoding 19385 chunks from 2985 documents in one batch...


Batches:   0%|          | 0/303 [00:00<?, ?it/s]

Encoding 638 chunks from 100 documents in one batch...


Batches:   0%|          | 0/10 [00:00<?, ?it/s]

Train embeddings: (2985, 384), Sample val embeddings: (100, 384)


### 2.2 Retrieve similar examples and build RAG prompts
> For each sampled document, find its 2 most similar training examples by cosine
similarity, then present them as worked examples in the same format as Section 5's
few-shot prompt the only difference is these examples are retrieved per-document
rather than fixed.

In [ ]:
def cosine_similarity_matrix(a, b):
    a_norm = a / np.linalg.norm(a, axis=1, keepdims=True)
    b_norm = b / np.linalg.norm(b, axis=1, keepdims=True)
    return a_norm @ b_norm.T

similarities = cosine_similarity_matrix(sample_val_embeddings, train_embeddings)

def get_similar_examples(doc_idx, k=2):
    top_k_idx = np.argsort(-similarities[doc_idx])[:k]
    return [(train_texts[i], train_labels[i]) for i in top_k_idx]


def build_rag_prompt(doc_text, candidate_ids, retrieved_examples):
    candidates_block = "\n".join(
        f"{i+1}. [{cid}] {concept_pool_lookup[cid]['term']}: {concept_pool_lookup[cid]['definition']}"
        for i, cid in enumerate(candidate_ids)
    )

    examples_block = ""
    for ex_text, ex_labels in retrieved_examples:
        concepts = [label['term'] for label in ex_labels]
        examples_block += f"\nExample passage: {ex_text[:300]}...\n"
        examples_block += f"Correct concepts (most to least relevant): {concepts}\n---\n"

    return f"""You are analysing a text to identify which social science concepts it implies,
even when those concepts are never explicitly named. The concepts are drawn from a formal
thesaurus and may be reflected through situations, actions, or themes in the text rather
than stated directly.

Here are examples retrieved for their similarity to the passage below:
{examples_block}

Now do the same for this new passage:

TEXT:
{doc_text}

CANDIDATE CONCEPTS:
{candidates_block}

Identify which of these candidate concepts are actually implied by the text, and rank
them from most to least relevant. Respond with ONLY a JSON array of concept IDs in ranked
order, e.g. ["id1", "id2", "id3"]. Include all 50 IDs, just reordered - do not omit any."""


import re

def parse_ranked_ids(response_text, valid_ids):
    match = re.search(r'\[.*\]', response_text, re.DOTALL)
    if not match:
        raise ValueError(f"No JSON array found in response: {response_text[:200]}")
    ranked = json.loads(match.group())
    valid_set = set(valid_ids)
    return [cid for cid in ranked if cid in valid_set]

### 2.3 Quick test before the full sample run

In [ ]:
test_idx = 0
test_doc_id = sample_val_ids[test_idx]
test_candidates = qwen_predictions[test_doc_id]
retrieved = get_similar_examples(test_idx)
test_prompt = build_rag_prompt(sample_val_texts[test_idx], test_candidates, retrieved)

response_text = call_llm(test_prompt)
ranked = parse_ranked_ids(response_text, test_candidates)

print(f"Gold: {[concept_pool_lookup[cid]['term'] for cid in gold[test_doc_id]]}")
print(f"Parsed {len(ranked)}/50 IDs")
print(f"Top 3: {[concept_pool_lookup[cid]['term'] for cid in ranked[:3]]}")

Gold: ['PRESIDENTIAL ELECTIONS', 'RESTORATIVE JUSTICE', 'FINANCIAL RESOURCES', 'CULTURAL EXPENDITURE', 'FESTIVALS']
Parsed 50/50 IDs
Top 3: ['PUBLIC EXPENDITURE', 'SOCIAL WELFARE EXPENDITURE', 'CULTURAL EXPENDITURE']


### 2.4 Full sample run (100 documents)

In [ ]:
def compute_standard_rag():
    checkpoint_path = os.path.join(CHECKPOINT_DIR, 'standard_rag_partial.json')
    failed_path = os.path.join(CHECKPOINT_DIR, 'standard_rag_failed_ids.json')
    predictions = {}
    failed_ids = []

    if os.path.exists(checkpoint_path):
        with open(checkpoint_path) as f:
            predictions = json.load(f)
        print(f"Resuming from {len(predictions)}/{len(sample_val_ids)} documents already done")

    for i, doc_id in enumerate(sample_val_ids):
        if doc_id in predictions:
            continue

        candidates = qwen_predictions[doc_id]
        retrieved = get_similar_examples(i)
        prompt = build_rag_prompt(sample_val_texts[i], candidates, retrieved)

        try:
            response_text = call_llm(prompt)
            ranked = parse_ranked_ids(response_text, candidates)
            predictions[doc_id] = ranked
            print(f"[{i+1}/{len(sample_val_ids)}] {doc_id} done")
        except Exception as e:
            print(f"Failed on {doc_id}: {e}")
            predictions[doc_id] = candidates
            failed_ids.append(doc_id)

        if i % 20 == 0:
            with open(checkpoint_path, 'w') as f:
                json.dump(predictions, f)
            with open(failed_path, 'w') as f:
                json.dump(failed_ids, f)
            print(f"Checkpoint saved: {len(predictions)}/{len(sample_val_ids)}")

    with open(checkpoint_path, 'w') as f:
        json.dump(predictions, f)
    with open(failed_path, 'w') as f:
        json.dump(failed_ids, f)

    # evaluate only on the 100-document sample's gold labels
    sample_gold = {doc_id: gold[doc_id] for doc_id in sample_val_ids}
    metrics = evaluate_retrieval(predictions, sample_gold)

    with open(os.path.join(RESULTS_DIR, 'standard_rag_predictions.json'), 'w') as f:
        json.dump(predictions, f)

    print(f"\nFinished. {len(failed_ids)}/{len(sample_val_ids)} documents needed fallback.")
    return metrics

standard_rag_metrics = run_or_load('standard_rag_retrieval', compute_standard_rag)
print(standard_rag_metrics)

Running: standard_rag_retrieval
[1/100] val_v00414 done
Checkpoint saved: 1/100
[2/100] val_v00090 done
[3/100] val_v00412 done
[4/100] val_v00597 done
[5/100] val_v00622 done
[6/100] val_v00069 done
[7/100] val_v00041 done
[8/100] val_v00101 done
[9/100] val_v00325 done
[10/100] val_v00489 done
[11/100] val_v00013 done
[12/100] val_v00352 done
[13/100] val_v00332 done
[14/100] val_v00544 done
[15/100] val_v00484 done
[16/100] val_v00337 done
[17/100] val_v00253 done
[18/100] val_v00465 done
[19/100] val_v00449 done
[20/100] val_v00536 done
[21/100] val_v00689 done
Checkpoint saved: 21/100
[22/100] val_v00704 done
[23/100] val_v00314 done
[24/100] val_v00478 done
[25/100] val_v00493 done
[26/100] val_v00423 done
[27/100] val_v00455 done
[28/100] val_v00341 done
[29/100] val_v00688 done
[30/100] val_v00184 done
[31/100] val_v00315 done
[32/100] val_v00230 done
[33/100] val_v00146 done
[34/100] val_v00141 done
[35/100] val_v00296 done
[36/100] val_v00302 done
[37/100] val_v00410 done
[38

### 2.5 Sample-restricted comparison & significance test
> Comparing all methods on the exact same 100-document sample (not the full-756 headline numbers) for a fair like-for-like comparison, then a Wilcoxon test to check whether Standard RAG's apparent edge over few-shot is real or within noise.

In [ ]:
with open(os.path.join(RESULTS_DIR, 'zero_shot_predictions.json')) as f:
    zero_shot_predictions = json.load(f)
with open(os.path.join(RESULTS_DIR, 'few_shot_predictions.json')) as f:
    few_shot_predictions = json.load(f)

sample_gold = {doc_id: gold[doc_id] for doc_id in sample_val_ids}
sample_qwen_preds = {doc_id: qwen_predictions[doc_id] for doc_id in sample_val_ids}
sample_zero_shot_preds = {doc_id: zero_shot_predictions[doc_id] for doc_id in sample_val_ids}
sample_few_shot_preds = {doc_id: few_shot_predictions[doc_id] for doc_id in sample_val_ids}

print("Same 100-document sample, for fair comparison:")
print(f"Qwen3 baseline:  {evaluate_retrieval(sample_qwen_preds, sample_gold)}")
print(f"Zero-shot:       {evaluate_retrieval(sample_zero_shot_preds, sample_gold)}")
print(f"Few-shot (fixed):{evaluate_retrieval(sample_few_shot_preds, sample_gold)}")
print(f"Standard RAG:    {standard_rag_metrics}")

Same 100-document sample, for fair comparison:
Qwen3 baseline:  {'MRR': 0.3890133219112999, 'Recall@5': 0.2781666666666666, 'Recall@10': 0.37633333333333335, 'NDCG@10': 0.3040808111536639}
Zero-shot:       {'MRR': 0.49891269841269836, 'Recall@5': 0.428, 'Recall@10': 0.5356666666666667, 'NDCG@10': 0.4356062280043641}
Few-shot (fixed):{'MRR': 0.5138677474486298, 'Recall@5': 0.4465, 'Recall@10': 0.5196666666666667, 'NDCG@10': 0.43548937199849325}
Standard RAG:    {'MRR': 0.5228651903651904, 'Recall@5': 0.45766666666666667, 'Recall@10': 0.5398333333333334, 'NDCG@10': 0.44833390295474146}


In [ ]:
from scipy.stats import wilcoxon

sample_few_shot_rr = [reciprocal_rank(sample_few_shot_preds[doc_id], sample_gold[doc_id]) for doc_id in sample_val_ids]

with open(os.path.join(RESULTS_DIR, 'standard_rag_predictions.json')) as f:
    standard_rag_predictions = json.load(f)
sample_rag_rr = [reciprocal_rank(standard_rag_predictions[doc_id], sample_gold[doc_id]) for doc_id in sample_val_ids]

stat, p = wilcoxon(sample_few_shot_rr, sample_rag_rr)
changed = sum(1 for doc_id in sample_val_ids if standard_rag_predictions[doc_id] != sample_few_shot_preds[doc_id])
print(f"Few-shot (fixed) vs Standard RAG (dynamic): {changed}/100 changed, p = {p:.4f}")

Few-shot (fixed) vs Standard RAG (dynamic): 100/100 changed, p = 0.3938


## 3. Ontology-Grounded RAG (OG-RAG)
> **Approach:** Rather than retrieving similar documents Standard RAG or relying on bare term+definition text zero-shot, this grounds each candidate concept with its actual ELSST hierarchy context its broader and narrower related concepts directly in the prompt. The model reasons with explicit ontology structure available, not just isolated definitions or embedding similarity.
>
> **Scope:** run on the full 756 validation documents, as the primary contribution of
this section.
>
> **Hypothesis:** Zero-shot and few-shot are statistically tied as the strongest prompting results so far (p=0.935); CoT performed significantly worse (p=0.0086) and Standard RAG's 100-document ablation showed a promising but not statistically confirmed trend above both. Given Section 7 found hierarchy structure alone only reaches parity with Qwen3 not exceeding it OG-RAG's genuine test is whether hierarchy context combined with LLM reasoning improves meaningfully on zero-shot's tied-best result, rather than assuming any
prior technique already holds a clear lead to beat.

In [ ]:
def format_hierarchy_context(cid):
    """Looks up a concept's broader/narrower ELSST neighbors and formats them as
    a short context string - kept brief since this gets added to all 50 candidates
    per prompt, and verbose context per candidate would blow up prompt length fast."""
    neighbor_ids = concept_neighbors.get(cid, set())
    if not neighbor_ids:
        return ""
    neighbor_terms = [concept_pool_lookup[nid]['term'] for nid in neighbor_ids if nid in concept_pool_lookup]
    if not neighbor_terms:
        return ""
    return f" (related in thesaurus to: {', '.join(neighbor_terms[:3])})"


def build_og_rag_prompt(doc_text, candidate_ids):
    candidates_block = "\n".join(
        f"{i+1}. [{cid}] {concept_pool_lookup[cid]['term']}: {concept_pool_lookup[cid]['definition']}"
        f"{format_hierarchy_context(cid)}"
        for i, cid in enumerate(candidate_ids)
    )

    return f"""You are analysing a text to identify which social science concepts it implies,
even when those concepts are never explicitly named. The concepts are drawn from a formal
thesaurus and may be reflected through situations, actions, or themes in the text rather
than stated directly.

Each candidate below includes its thesaurus context - related broader/narrower concepts -
which can help you judge how specific or general a match actually is, and whether a cluster
of related concepts all being relevant strengthens the case for one of them.

Passage:
{doc_text}

Candidate concepts:
{candidates_block}

Identify which of these candidate concepts are actually implied by the text, and rank
them from most to least relevant. Respond with ONLY a JSON array of concept IDs in ranked
order, e.g. ["id1", "id2", "id3"]. Include all 50 IDs, just reordered - do not omit any."""

### 3.1 Quick test before the full run

In [ ]:
test_doc_id = val_ids[0]
test_idx = val_ids.index(test_doc_id)
test_candidates = qwen_predictions[test_doc_id]
test_prompt = build_og_rag_prompt(val_texts[test_idx], test_candidates)

response_text = call_llm(test_prompt)
ranked = parse_ranked_ids(response_text, test_candidates)

print(f"Gold: {[concept_pool_lookup[cid]['term'] for cid in gold[test_doc_id]]}")
print(f"Parsed {len(ranked)}/50 IDs")
print(f"Top 3: {[concept_pool_lookup[cid]['term'] for cid in ranked[:3]]}")

Gold: ['REGIONAL ECONOMY', 'REGIONAL FINANCE']
Parsed 50/50 IDs
Top 3: ['REGIONAL FINANCE', 'REGIONAL ECONOMY', 'LOCAL FINANCE']


### 3.2 Full validation run

In [ ]:
def compute_og_rag():
    checkpoint_path = os.path.join(CHECKPOINT_DIR, 'og_rag_partial.json')
    failed_path = os.path.join(CHECKPOINT_DIR, 'og_rag_failed_ids.json')
    predictions = {}
    failed_ids = []

    if os.path.exists(checkpoint_path):
        with open(checkpoint_path) as f:
            predictions = json.load(f)
        print(f"Resuming from {len(predictions)}/{len(val_ids)} documents already done")

    for i, doc_id in enumerate(val_ids):
        if doc_id in predictions:
            continue

        idx = val_ids.index(doc_id)
        candidates = qwen_predictions[doc_id]
        prompt = build_og_rag_prompt(val_texts[idx], candidates)

        try:
            response_text = call_llm(prompt)
            ranked = parse_ranked_ids(response_text, candidates)
            predictions[doc_id] = ranked
            print(f"[{i+1}/{len(val_ids)}] {doc_id} done")
        except Exception as e:
            print(f"Failed on {doc_id}: {e}")
            predictions[doc_id] = candidates
            failed_ids.append(doc_id)

        if i % 20 == 0:
            with open(checkpoint_path, 'w') as f:
                json.dump(predictions, f)
            with open(failed_path, 'w') as f:
                json.dump(failed_ids, f)
            print(f"Checkpoint saved: {len(predictions)}/{len(val_ids)} ({len(failed_ids)} failed so far)")

    with open(checkpoint_path, 'w') as f:
        json.dump(predictions, f)
    with open(failed_path, 'w') as f:
        json.dump(failed_ids, f)

    metrics = evaluate_retrieval(predictions, gold)

    with open(os.path.join(RESULTS_DIR, 'og_rag_predictions.json'), 'w') as f:
        json.dump(predictions, f)

    print(f"\nFinished. {len(failed_ids)}/{len(val_ids)} documents needed fallback.")
    return metrics

og_rag_metrics = run_or_load('og_rag_retrieval', compute_og_rag)
print(og_rag_metrics)

Running: og_rag_retrieval
[1/756] val_v00029 done
Checkpoint saved: 1/756 (0 failed so far)
[2/756] val_v00010 done
[3/756] val_v00002 done
[4/756] val_v00015 done
[5/756] val_v00035 done
[6/756] val_v00004 done
[7/756] val_v00038 done
[8/756] val_v00001 done
[9/756] val_v00030 done
[10/756] val_v00028 done
[11/756] val_v00003 done
[12/756] val_v00005 done
[13/756] val_v00025 done
[14/756] val_v00040 done
[15/756] val_v00011 done
[16/756] val_v00019 done
[17/756] val_v00037 done
[18/756] val_v00012 done
[19/756] val_v00031 done
[20/756] val_v00026 done
[21/756] val_v00039 done
Checkpoint saved: 21/756 (0 failed so far)
[22/756] val_v00023 done
[23/756] val_v00036 done
[24/756] val_v00018 done
[25/756] val_v00014 done
[26/756] val_v00022 done
[27/756] val_v00021 done
[28/756] val_v00032 done
[29/756] val_v00007 done
[30/756] val_v00016 done
[31/756] val_v00020 done
[32/756] val_v00008 done
[33/756] val_v00024 done
[34/756] val_v00034 done
[35/756] val_v00009 done
[36/756] val_v00006 don

### 3.3 Retry the failed document
> One document failed with a transient 503 error during the full run - retrying it individually rather than leaving it on Qwen3's raw fallback order.

In [ ]:
with open(os.path.join(CHECKPOINT_DIR, 'og_rag_failed_ids.json')) as f:
    failed_ids = json.load(f)
with open(os.path.join(RESULTS_DIR, 'og_rag_predictions.json')) as f:
    predictions = json.load(f)

In [ ]:
if failed_ids:
    failed_doc_id = failed_ids[0]
    print(f"Retrying: {failed_doc_id}")

    idx = val_ids.index(failed_doc_id)
    candidates = qwen_predictions[failed_doc_id]
    prompt = build_og_rag_prompt(val_texts[idx], candidates)

    try:
        response_text = call_llm(prompt)
        ranked = parse_ranked_ids(response_text, candidates)
        predictions[failed_doc_id] = ranked
        print(f"Recovered: {failed_doc_id}")
    except Exception as e:
        print(f"Still failing: {e}")

    metrics = evaluate_retrieval(predictions, gold)
    with open(os.path.join(RESULTS_DIR, 'og_rag_predictions.json'), 'w') as f:
        json.dump(predictions, f)
    with open(os.path.join(RESULTS_DIR, 'og_rag_retrieval.json'), 'w') as f:
        json.dump(metrics, f, indent=2)

    print(metrics)
else:
    print("No failed_ids found in memory - need to reload from checkpoint file")

Retrying: val_v00277
Recovered: val_v00277
{'MRR': 0.49695893552102705, 'Recall@5': 0.3947751322751323, 'Recall@10': 0.5018518518518518, 'NDCG@10': 0.4047984737733954}


### 3.4 Statistical significance vs. zero-shot
> Checking whether OG-RAG's small apparent difference from zero-shot is a real effect or noise, same paired Wilcoxon approach as before.

In [ ]:
with open(os.path.join(RESULTS_DIR, 'zero_shot_predictions.json')) as f:
    zero_shot_predictions_check = json.load(f)

In [ ]:
og_rag_rr = [reciprocal_rank(predictions[doc_id], gold[doc_id]) for doc_id in val_ids]
zero_shot_rr_check = [reciprocal_rank(zero_shot_predictions_check[doc_id], gold[doc_id]) for doc_id in val_ids]

stat, p = wilcoxon(zero_shot_rr_check, og_rag_rr)
changed = sum(1 for doc_id in val_ids if predictions[doc_id] != zero_shot_predictions_check[doc_id])
print(f"Zero-shot vs OG-RAG: {changed}/756 changed, p = {p:.4f}")

Zero-shot vs OG-RAG: 756/756 changed, p = 0.2722


## 4. Novel Pipeline: Selective Hierarchy-Aware Reasoning
> **Motivation:** Section 7 found that raw hierarchy signals fail because most of Qwen3's top-50 candidates are already thematically related to each other. But one specific failure mode was real and diagnosable: broad concepts (e.g. HOUSING) are crowding out their more precise relatives (e.g. HIGH RISE FLATS). Section 6 also found that forcing exhaustive reasoning CoT on every document dilutes the model's focus once confident judgments run out.
>
> **Approach:** Detect documents where the top-50 candidates contain a genuine broader/narrower pair a real risk of the HOUSING/HIGH RISE FLATS confusion. For those documents only, prompt the model to reason specifically about that pair. For all other documents, use a simple prompt like zero-shot, avoiding CoT's over-application problem.
>
> **First step:** check how common these conflicts actually are, before committing to a full run if every document has one, this isn't meaningfully "selective."

In [ ]:
def find_hierarchy_conflicts(candidates):
    """Finds broader/narrower pairs within a single document's candidate list -
    these are the 'confusable pairs' risking the HOUSING/HIGH RISE FLATS mistake
    identified in Section 7. Uses the strict (broader/narrower only) neighbor set,
    same as Section 7's best-performing hierarchy variant."""
    candidate_set = set(candidates)
    seen_pairs = set()
    conflicts = []
    for cid in candidates:
        neighbors = concept_neighbors.get(cid, set())
        for nid in neighbors & candidate_set:
            pair = tuple(sorted([cid, nid]))
            if pair not in seen_pairs:
                seen_pairs.add(pair)
                conflicts.append(pair)
    return conflicts

conflict_counts = {doc_id: len(find_hierarchy_conflicts(qwen_predictions[doc_id])) for doc_id in val_ids}

zero_conflict = sum(1 for c in conflict_counts.values() if c == 0)
one_conflict = sum(1 for c in conflict_counts.values() if c == 1)
multi_conflict = sum(1 for c in conflict_counts.values() if c > 1)

print(f"Documents with 0 conflicts: {zero_conflict}/756")
print(f"Documents with 1 conflict: {one_conflict}/756")
print(f"Documents with 2+ conflicts: {multi_conflict}/756")
print(f"Average conflicts per document: {sum(conflict_counts.values()) / len(conflict_counts):.2f}")

Documents with 0 conflicts: 0/756
Documents with 1 conflict: 5/756
Documents with 2+ conflicts: 751/756
Average conflicts per document: 12.65


In [ ]:
def find_top_k_conflicts(candidates, k=10):
    """Same conflict detection as before, but restricted to the top-k candidates -
    where a HOUSING/HIGH RISE FLATS-style mix-up would actually change the ranking
    that matters for MRR and Recall@5, rather than noise buried near the bottom."""
    top_k = candidates[:k]
    top_k_set = set(top_k)
    seen_pairs = set()
    conflicts = []
    for cid in top_k:
        neighbors = concept_neighbors.get(cid, set())
        for nid in neighbors & top_k_set:
            pair = tuple(sorted([cid, nid]))
            if pair not in seen_pairs:
                seen_pairs.add(pair)
                conflicts.append(pair)
    return conflicts

top10_conflict_counts = {doc_id: len(find_top_k_conflicts(qwen_predictions[doc_id], k=10)) for doc_id in val_ids}

zero_conflict = sum(1 for c in top10_conflict_counts.values() if c == 0)
has_conflict = sum(1 for c in top10_conflict_counts.values() if c >= 1)

print(f"Documents with 0 top-10 conflicts: {zero_conflict}/756")
print(f"Documents with 1+ top-10 conflicts: {has_conflict}/756")
print(f"Average top-10 conflicts per document: {sum(top10_conflict_counts.values()) / len(top10_conflict_counts):.2f}")

Documents with 0 top-10 conflicts: 268/756
Documents with 1+ top-10 conflicts: 488/756
Average top-10 conflicts per document: 1.33


### 4.1 Selective prompt construction
> Documents with a top-10 hierarchy conflict get a prompt that explicitly flags the conflicting pair and asks the model to judge specificity carefully between them a
targeted nudge, not exhaustive reasoning over all 50 candidates. Documents with no
conflict get a plain prompt, avoiding CoT's over-application problem entirely.

In [ ]:
def build_plain_prompt(doc_text, candidate_ids):
    """Same style as zero-shot (Section 4) - used when no hierarchy conflict exists,
    since there's nothing specific to disambiguate."""
    candidates_block = "\n".join(
        f"{i+1}. [{cid}] {concept_pool_lookup[cid]['term']}: {concept_pool_lookup[cid]['definition']}"
        for i, cid in enumerate(candidate_ids)
    )
    return f"""You are analysing a text to identify which social science concepts it implies,
even when those concepts are never explicitly named.

Passage:
{doc_text}

Candidate concepts:
{candidates_block}

Identify which of these candidate concepts are actually implied by the text, and rank
them from most to least relevant. Respond with ONLY a JSON array of concept IDs in ranked
order, e.g. ["id1", "id2", "id3"]. Include all 50 IDs, just reordered - do not omit any."""


def build_selective_prompt(doc_text, candidate_ids):
    """Used when a top-10 hierarchy conflict exists - flags the specific confusable
    pair(s) and asks for a targeted judgment call, rather than asking the model to
    reason step-by-step over the entire list (the approach that failed in Section 6)."""
    conflicts = find_top_k_conflicts(candidate_ids, k=10)
    conflict_lines = "\n".join(
        f"- \"{concept_pool_lookup[a]['term']}\" vs \"{concept_pool_lookup[b]['term']}\""
        for a, b in conflicts
    )

    candidates_block = "\n".join(
        f"{i+1}. [{cid}] {concept_pool_lookup[cid]['term']}: {concept_pool_lookup[cid]['definition']}"
        for i, cid in enumerate(candidate_ids)
    )

    return f"""You are analysing a text to identify which social science concepts it implies,
even when those concepts are never explicitly named.

This candidate list contains related but distinct concepts that are easy to confuse - a
broader, more general concept alongside a narrower, more specific one:
{conflict_lines}

Before ranking, briefly consider for each pair above: does the text point to the general
concept, or specifically to the narrower one? A text about a flood-damaged apartment block
points to the specific concept, not the broad category it belongs to.

Passage:
{doc_text}

Candidate concepts:
{candidates_block}

Now rank all candidates from most to least relevant, applying that judgment where a
conflicting pair is involved. Respond with ONLY a JSON array of concept IDs in ranked
order, e.g. ["id1", "id2", "id3"]. Include all 50 IDs, just reordered - do not omit any."""


def build_novel_prompt(doc_text, candidate_ids):
    conflicts = find_top_k_conflicts(candidate_ids, k=10)
    if conflicts:
        return build_selective_prompt(doc_text, candidate_ids), True
    return build_plain_prompt(doc_text, candidate_ids), False

### 4.2 Quick test - one document from each path

In [ ]:
conflict_doc_id = next(doc_id for doc_id in val_ids if top10_conflict_counts[doc_id] >= 1)
no_conflict_doc_id = next(doc_id for doc_id in val_ids if top10_conflict_counts[doc_id] == 0)

for label, doc_id in [("WITH conflict", conflict_doc_id), ("NO conflict", no_conflict_doc_id)]:
    idx = val_ids.index(doc_id)
    candidates = qwen_predictions[doc_id]
    prompt, used_selective = build_novel_prompt(val_texts[idx], candidates)

    response_text = call_llm(prompt)
    ranked = parse_ranked_ids(response_text, candidates)

    print(f"--- {label} ({doc_id}), used_selective={used_selective} ---")
    print(f"Gold: {[concept_pool_lookup[cid]['term'] for cid in gold[doc_id]]}")
    print(f"Parsed {len(ranked)}/50 IDs")
    print(f"Top 3: {[concept_pool_lookup[cid]['term'] for cid in ranked[:3]]}\n")

--- WITH conflict (val_v00029), used_selective=True ---
Gold: ['REGIONAL ECONOMY', 'REGIONAL FINANCE']
Parsed 50/50 IDs
Top 3: ['REGIONAL FINANCE', 'REGIONAL ECONOMY', 'DECENTRALIZATION']

--- NO conflict (val_v00002), used_selective=False ---
Gold: ['SINGLE-SEX SCHOOLS', 'RESEARCH CENTRES']
Parsed 48/50 IDs
Top 3: ['SINGLE-SEX SCHOOLS', 'EDUCATIONAL RESEARCH', 'EVALUATION OF EDUCATION']



### 4.3 Parsing Completeness Fix
> While testing the novel pipeline's routing, noticed some documents parsed to fewer than 50 ranked IDs - `parse_ranked_ids` was silently dropping any candidate ID the model's JSON response omitted, rather than padding it back in like the CoT parser does. Checking severity directly (did a dropped ID ever happen to be a gold concept) before deciding whether this needs fixing, rather than assuming from missing-ID counts alone.

In [ ]:
def check_parse_completeness(predictions_dict, label):
    short_docs = [doc_id for doc_id, ranked in predictions_dict.items() if len(ranked) < 50]
    print(f"{label}: {len(short_docs)}/{len(predictions_dict)} documents with <50 IDs")
    if short_docs:
        lengths = [len(predictions_dict[d]) for d in short_docs]
        print(f"  Shortest: {min(lengths)}, average length of short ones: {sum(lengths)/len(lengths):.1f}")

with open(os.path.join(RESULTS_DIR, 'zero_shot_predictions.json')) as f:
    check_parse_completeness(json.load(f), "Zero-shot")
with open(os.path.join(RESULTS_DIR, 'few_shot_predictions.json')) as f:
    check_parse_completeness(json.load(f), "Few-shot")
with open(os.path.join(RESULTS_DIR, 'og_rag_predictions.json')) as f:
    check_parse_completeness(json.load(f), "OG-RAG")

Zero-shot: 160/756 documents with <50 IDs
  Shortest: 0, average length of short ones: 48.4
Few-shot: 82/756 documents with <50 IDs
  Shortest: 34, average length of short ones: 48.7
OG-RAG: 153/756 documents with <50 IDs
  Shortest: 35, average length of short ones: 48.8


In [ ]:
def check_gold_impact(predictions_dict, label):
    affected_docs = []
    for doc_id, ranked in predictions_dict.items():
        if len(ranked) >= 50:
            continue
        original_candidates = set(qwen_predictions[doc_id])
        parsed_set = set(ranked)
        dropped = original_candidates - parsed_set
        gold_set = set(gold[doc_id])
        lost_gold = dropped & gold_set
        if lost_gold:
            affected_docs.append((doc_id, lost_gold))

    print(f"{label}: {len(affected_docs)} documents where a GOLD concept was actually dropped")
    return affected_docs

with open(os.path.join(RESULTS_DIR, 'zero_shot_predictions.json')) as f:
    zs_affected = check_gold_impact(json.load(f), "Zero-shot")
with open(os.path.join(RESULTS_DIR, 'few_shot_predictions.json')) as f:
    fs_affected = check_gold_impact(json.load(f), "Few-shot")
with open(os.path.join(RESULTS_DIR, 'og_rag_predictions.json')) as f:
    og_affected = check_gold_impact(json.load(f), "OG-RAG")

Zero-shot: 2 documents where a GOLD concept was actually dropped
Few-shot: 1 documents where a GOLD concept was actually dropped
OG-RAG: 3 documents where a GOLD concept was actually dropped


In [ ]:
def get_affected_ids(predictions_dict):
    affected = []
    for doc_id, ranked in predictions_dict.items():
        if len(ranked) >= 50:
            continue
        dropped = set(qwen_predictions[doc_id]) - set(ranked)
        if dropped & set(gold[doc_id]):
            affected.append(doc_id)
    return affected

with open(os.path.join(RESULTS_DIR, 'zero_shot_predictions.json')) as f:
    zero_shot_predictions = json.load(f)
with open(os.path.join(RESULTS_DIR, 'few_shot_predictions.json')) as f:
    few_shot_predictions = json.load(f)
with open(os.path.join(RESULTS_DIR, 'og_rag_predictions.json')) as f:
    og_rag_predictions = json.load(f)

zs_ids = get_affected_ids(zero_shot_predictions)
fs_ids = get_affected_ids(few_shot_predictions)
og_ids = get_affected_ids(og_rag_predictions)
print(zs_ids, fs_ids, og_ids)

['val_v00378', 'val_v00695'] ['val_v00351'] ['val_v00224', 'val_v00262', 'val_v00488']


#### 4.3.1 Patching OG-RAG
> 3 documents had a gold concept dropped by the parsing gap re-calling only those, then re-evaluating.

In [ ]:
for doc_id in og_ids:
    idx = val_ids.index(doc_id)
    prompt = build_og_rag_prompt(val_texts[idx], qwen_predictions[doc_id])
    response_text = call_llm(prompt)
    og_rag_predictions[doc_id] = parse_ranked_ids(response_text, qwen_predictions[doc_id])

with open(os.path.join(RESULTS_DIR, 'og_rag_predictions.json'), 'w') as f:
    json.dump(og_rag_predictions, f)
print(evaluate_retrieval(og_rag_predictions, gold))

{'MRR': 0.4974025248535053, 'Recall@5': 0.39510582010582007, 'Recall@10': 0.5025132275132275, 'NDCG@10': 0.40523473574137925}


#### 4.3.2 Patching zero-shot
> 2 documents affected same targeted fix.

In [ ]:
def build_zero_shot_prompt(doc_text, candidate_ids):
    candidates_block = "\n".join(
        f"{i+1}. [{cid}] {concept_pool_lookup[cid]['term']}: {concept_pool_lookup[cid]['definition']}"
        for i, cid in enumerate(candidate_ids)
    )
    return f"""You are analysing a text to identify which social science concepts it implies,
even when those concepts are never explicitly named.

Passage:
{doc_text}

Candidate concepts:
{candidates_block}

Identify which of these candidate concepts are actually implied by the text, and rank
them from most to least relevant. Respond with ONLY a JSON array of concept IDs in ranked
order, e.g. ["id1", "id2", "id3"]. Include all 50 IDs, just reordered - do not omit any."""

In [ ]:
for doc_id in zs_ids:
    idx = val_ids.index(doc_id)
    prompt = build_zero_shot_prompt(val_texts[idx], qwen_predictions[doc_id])
    response_text = call_llm(prompt)
    zero_shot_predictions[doc_id] = parse_ranked_ids(response_text, qwen_predictions[doc_id])

with open(os.path.join(RESULTS_DIR, 'zero_shot_predictions.json'), 'w') as f:
    json.dump(zero_shot_predictions, f)
print(evaluate_retrieval(zero_shot_predictions, gold))

{'MRR': 0.5020693362906619, 'Recall@5': 0.41040564373897703, 'Recall@10': 0.5080026455026455, 'NDCG@10': 0.41256958436652225}


#### 4.3.3 Patching few-shot
> Only 1 document affected. Reconstructing the exact original worked examples the first single-concept and first 4+-concept training documents so the patch uses an identical prompt to the original run.

In [ ]:
train_rows = dataset['train']
example_single = next(row for row in train_rows if len(row['generation_labels']) == 1)
example_multi = next(row for row in train_rows if len(row['generation_labels']) >= 4)
few_shot_examples = [example_single, example_multi]

In [ ]:
def build_few_shot_prompt(doc_text, candidate_ids, examples):
    candidates_block = "\n".join(
        f"{i+1}. [{cid}] {concept_pool_lookup[cid]['term']}: {concept_pool_lookup[cid]['definition']}"
        for i, cid in enumerate(candidate_ids)
    )

    examples_block = ""
    for ex in examples:
        concepts = [label['term'] for label in ex['generation_labels']]
        examples_block += f"\nExample passage: {ex['text'][:300]}...\n"
        examples_block += f"Correct concepts (most to least relevant): {concepts}\n---\n"

    return f"""You are analysing a text to identify which social science concepts it implies,
even when those concepts are never explicitly named. The concepts are drawn from a formal
thesaurus and may be reflected through situations, actions, or themes in the text rather
than stated directly.

Here are some worked examples:
{examples_block}

Now do the same for this new passage:

TEXT:
{doc_text}

CANDIDATE CONCEPTS:
{candidates_block}

Identify which of these candidate concepts are actually implied by the text, and rank
them from most to least relevant. Respond with ONLY a JSON array of concept IDs in ranked
order, e.g. ["id1", "id2", "id3"]. Include all 50 IDs, just reordered - do not omit any."""

In [ ]:
for doc_id in fs_ids:
    idx = val_ids.index(doc_id)
    prompt = build_few_shot_prompt(val_texts[idx], qwen_predictions[doc_id], few_shot_examples)
    response_text = call_llm(prompt)
    few_shot_predictions[doc_id] = parse_ranked_ids(response_text, qwen_predictions[doc_id])

with open(os.path.join(RESULTS_DIR, 'few_shot_predictions.json'), 'w') as f:
    json.dump(few_shot_predictions, f)
print(evaluate_retrieval(few_shot_predictions, gold))

{'MRR': 0.5069864636595559, 'Recall@5': 0.40694444444444444, 'Recall@10': 0.5032186948853615, 'NDCG@10': 0.4103958652195154}


### 4.4 Full validation run

In [ ]:
def compute_novel_pipeline():
    checkpoint_path = os.path.join(CHECKPOINT_DIR, 'novel_pipeline_partial.json')
    failed_path = os.path.join(CHECKPOINT_DIR, 'novel_pipeline_failed_ids.json')
    selective_flags_path = os.path.join(CHECKPOINT_DIR, 'novel_pipeline_selective_flags.json')
    predictions = {}
    failed_ids = []
    selective_flags = {}

    if os.path.exists(checkpoint_path):
        with open(checkpoint_path) as f:
            predictions = json.load(f)
        with open(selective_flags_path) as f:
            selective_flags = json.load(f)
        print(f"Resuming from {len(predictions)}/{len(val_ids)} documents already done")

    for i, doc_id in enumerate(val_ids):
        if doc_id in predictions:
            continue

        idx = val_ids.index(doc_id)
        candidates = qwen_predictions[doc_id]
        prompt, used_selective = build_novel_prompt(val_texts[idx], candidates)

        try:
            response_text = call_llm(prompt)
            ranked = parse_ranked_ids(response_text, candidates)
            predictions[doc_id] = ranked
            selective_flags[doc_id] = used_selective
            print(f"[{i+1}/{len(val_ids)}] {doc_id} done (selective={used_selective})")
        except Exception as e:
            print(f"Failed on {doc_id}: {e}")
            predictions[doc_id] = candidates
            selective_flags[doc_id] = used_selective
            failed_ids.append(doc_id)

        if i % 20 == 0:
            with open(checkpoint_path, 'w') as f:
                json.dump(predictions, f)
            with open(failed_path, 'w') as f:
                json.dump(failed_ids, f)
            with open(selective_flags_path, 'w') as f:
                json.dump(selective_flags, f)
            print(f"Checkpoint saved: {len(predictions)}/{len(val_ids)} ({len(failed_ids)} failed so far)")

    with open(checkpoint_path, 'w') as f:
        json.dump(predictions, f)
    with open(failed_path, 'w') as f:
        json.dump(failed_ids, f)
    with open(selective_flags_path, 'w') as f:
        json.dump(selective_flags, f)

    metrics = evaluate_retrieval(predictions, gold)

    with open(os.path.join(RESULTS_DIR, 'novel_pipeline_predictions.json'), 'w') as f:
        json.dump(predictions, f)

    print(f"\nFinished. {len(failed_ids)}/{len(val_ids)} documents needed fallback.")
    print(f"Used selective reasoning on {sum(selective_flags.values())}/{len(selective_flags)} documents.")
    return metrics

novel_pipeline_metrics = run_or_load('novel_pipeline_retrieval', compute_novel_pipeline)
print(novel_pipeline_metrics)

Running: novel_pipeline_retrieval
[1/756] val_v00029 done (selective=True)
Checkpoint saved: 1/756 (0 failed so far)
[2/756] val_v00010 done (selective=True)
[3/756] val_v00002 done (selective=False)
[4/756] val_v00015 done (selective=False)
[5/756] val_v00035 done (selective=True)
[6/756] val_v00004 done (selective=True)
[7/756] val_v00038 done (selective=True)
[8/756] val_v00001 done (selective=True)
[9/756] val_v00030 done (selective=False)
[10/756] val_v00028 done (selective=True)
[11/756] val_v00003 done (selective=True)
[12/756] val_v00005 done (selective=True)
[13/756] val_v00025 done (selective=False)
[14/756] val_v00040 done (selective=True)
[15/756] val_v00011 done (selective=True)
[16/756] val_v00019 done (selective=False)
[17/756] val_v00037 done (selective=True)
[18/756] val_v00012 done (selective=True)
[19/756] val_v00031 done (selective=True)
[20/756] val_v00026 done (selective=True)
[21/756] val_v00039 done (selective=False)
Checkpoint saved: 21/756 (0 failed so far)
[2

### 4.5 Retry the failed documents
> 5 documents failed with genuine malformed-JSON errors during the full run checking each one's fallback score first some had real room to improve, one was already perfect, then retrying all 5.

In [ ]:
with open(os.path.join(CHECKPOINT_DIR, 'novel_pipeline_failed_ids.json')) as f:
    novel_failed_ids = json.load(f)

print(f"Failed documents: {novel_failed_ids}")

for doc_id in novel_failed_ids:
    fallback_rank = reciprocal_rank(predictions[doc_id], gold[doc_id])
    print(f"{doc_id}: fallback reciprocal rank = {fallback_rank:.3f}")

Failed documents: ['val_v00134', 'val_v00228', 'val_v00293', 'val_v00361', 'val_v00484']
val_v00134: fallback reciprocal rank = 0.500
val_v00228: fallback reciprocal rank = 0.000
val_v00293: fallback reciprocal rank = 0.333
val_v00361: fallback reciprocal rank = 0.023
val_v00484: fallback reciprocal rank = 1.000


In [ ]:
for doc_id in novel_failed_ids:
    idx = val_ids.index(doc_id)
    candidates = qwen_predictions[doc_id]
    prompt, used_selective = build_novel_prompt(val_texts[idx], candidates)

    try:
        response_text = call_llm(prompt)
        ranked = parse_ranked_ids(response_text, candidates)
        predictions[doc_id] = ranked
        selective_flags[doc_id] = used_selective
        print(f"Recovered: {doc_id} (selective={used_selective})")
    except Exception as e:
        print(f"Still failing: {doc_id}: {e}")

with open(os.path.join(RESULTS_DIR, 'novel_pipeline_predictions.json'), 'w') as f:
    json.dump(predictions, f)

metrics = evaluate_retrieval(predictions, gold)
print(metrics)

Recovered: val_v00134 (selective=True)
Recovered: val_v00228 (selective=False)
Recovered: val_v00293 (selective=False)
Recovered: val_v00361 (selective=True)
Recovered: val_v00484 (selective=True)
{'MRR': 0.48994213568991374, 'Recall@5': 0.3950396825396825, 'Recall@10': 0.49369488536155204, 'NDCG@10': 0.4008235759992664}


### 4.6 Re-verifying the segmented comparison
> Re-running the conflict-subset vs. plain subset comparison against zero-shot now that all 756 documents have a genuine model generated ranking, to confirm the earlier finding conflict subset ties with zero-shot; plain subset's apparent difference is API non-determinism still holds after the retry.

In [ ]:
with open(os.path.join(CHECKPOINT_DIR, 'novel_pipeline_selective_flags.json')) as f:
    selective_flags = json.load(f)

with open(os.path.join(RESULTS_DIR, 'novel_pipeline_predictions.json')) as f:
    predictions = json.load(f)

In [ ]:
selective_ids = [doc_id for doc_id, used in selective_flags.items() if used]
plain_ids = [doc_id for doc_id, used in selective_flags.items() if not used]

for label, ids in [("Selective/conflict subset", selective_ids), ("Plain subset", plain_ids)]:
    novel_subset = {doc_id: predictions[doc_id] for doc_id in ids}
    zero_shot_subset = {doc_id: zero_shot_predictions[doc_id] for doc_id in ids}
    gold_subset = {doc_id: gold[doc_id] for doc_id in ids}

    novel_metrics = evaluate_retrieval(novel_subset, gold_subset)
    zs_metrics = evaluate_retrieval(zero_shot_subset, gold_subset)

    novel_rr = [reciprocal_rank(novel_subset[d], gold_subset[d]) for d in ids]
    zs_rr = [reciprocal_rank(zero_shot_subset[d], gold_subset[d]) for d in ids]
    stat, p = wilcoxon(zs_rr, novel_rr)

    print(f" {label} (n={len(ids)}) ")
    print(f"Novel pipeline: {novel_metrics}")
    print(f"Zero-shot:      {zs_metrics}")
    print(f"Wilcoxon p = {p:.4f}\n")

 Selective/conflict subset (n=488) 
Novel pipeline: {'MRR': 0.5183995891522015, 'Recall@5': 0.4342896174863388, 'Recall@10': 0.5301571038251367, 'NDCG@10': 0.435475493619565}
Zero-shot:      {'MRR': 0.522386959062908, 'Recall@5': 0.44422814207650274, 'Recall@10': 0.5413251366120218, 'NDCG@10': 0.4396528763734859}
Wilcoxon p = 0.5206

 Plain subset (n=268) 
Novel pipeline: {'MRR': 0.43812408610186715, 'Recall@5': 0.32356965174129354, 'Recall@10': 0.42730099502487556, 'NDCG@10': 0.33772605436230474}
Zero-shot:      {'MRR': 0.46507306795910924, 'Recall@5': 0.34881840796019903, 'Recall@10': 0.44732587064676615, 'NDCG@10': 0.3632537392195139}
Wilcoxon p = 0.0436

